# From py4dgeo to HDF5

Convert SpatiotemporalAnalysis.zip from py4dgeo to a portable analysis ready h5-based file.

Core features:
- point × time storage
- xarray-compatible HDF5
- sparse OBC membership
- resumable conversion
- immutable finished HDF5 product


Workflow:
```
py4dgeo SpatiotemporalAnalysis.zip
        ↓
      obc_h5.py
        ↓
      dataset.h5
        ├── /analysis
        └── /obcs
```

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from topol.io.obc_h5 import (
    ConversionConfig,
    convert_py4dgeo_zip,
    OBCH5,
)
# `# pip install -e .`, if import fails

from shapely.geometry import box, Polygon

## 2. Define input and output paths

In [3]:
# Original py4dgeo SpatiotemporalAnalysis zip.
source_zip = Path(
    "/media/ssd01/jiapan/repos/TopoL/0_data/derived/py4dgeo_analysis/Fin_riverbank/riverbank_4dobc_5cm_cylrad5cm_normalrad20cm50cm_rdp.zip"
)

# Final HDF5 representation.
output_h5 = Path(
    "/media/ssd01/jiapan/repos/TopoL/0_data/TopoL-4D/FinRiverbank/h5/FinRiverbank_rdp.h5"
)

## 3. Configs

| Option | Purpose | Recommended default |
|---|---|---|
| `dataset_id` | Stable dataset/site identifier | Required |
| `crs` | CRS of core-point coordinates | Set explicitly |
| `point_chunk` | Physical HDF5 point chunk size | `512` |
| `write_block_points` | Number of points written per conversion block | `8192` |
| `compression_level` | gzip compression level | `1` |
| `include_smoothed_distances` | Store py4dgeo smoothed time series | `True` if used |
| `uncertainty_fields` | Uncertainty variables to preserve | `("lodetection",)`,  ``None`` for all fields or ``()`` for none |
| `include_obc_distances` | Store OBC region-growing distances (i.e., DTW distance) | `False` |
| `include_reverse_index` | Store point → OBC lookup index | `False` |
| `checkpoint_every_obcs` | Flush/checkpoint interval during OBC writing | `25` |

In [3]:
config = ConversionConfig(
    dataset_id="FinRiverbank",
    crs="EPSG:25832",

    point_chunk=512,
    write_block_points=8192,
    compression_level=1,

    include_smoothed_distances=True,

    uncertainty_fields=None,

    include_obc_distances=True,
    include_reverse_index=True,

    checkpoint_every_obcs=25,
)

config

ConversionConfig(dataset_id='FinRiverbank', crs='EPSG:25832', distance_units='m', point_chunk=512, time_chunk=None, write_block_points=8192, compression_level=1, include_smoothed_distances=True, uncertainty_fields=None, include_obc_distances=True, include_reverse_index=True, checkpoint_every_obcs=25)

## 4. Conversion

If conversion is interrupted, run the same cell again.

The converter uses: `FinRiverbank.partial.h5` as a checkpoint file and resumes from completed blocks.

Do not use overwrite=True unless you intentionally want to restart the conversion from the beginning.

In [4]:
## create the HDF5 file from the py4dgeo zip file
convert_py4dgeo_zip(
    source_zip,
    output_h5,
    config=config,
    overwrite=False, # False by default, set to True to overwrite existing HDF5 file
)

[2026-10-07 10:30:14][INFO] Restoring epoch from file '/tmp/tmpodzs5au3/corepoints.zip'
[2026-10-07 10:30:14][INFO] Restoring epoch from file '/tmp/tmpoeppijmc/reference_epoch.zip'


PosixPath('/media/ssd01/jiapan/repos/TopoL/0_data/TopoL-4D/FinRiverbank/h5/FinRiverbank_rdp_merged.h5')

## 5. Inspect the generated HDF5

Now you can read the h5 via xarray.

In [5]:
analysis = xr.open_dataset(
    output_h5,
    engine="h5netcdf",
    group="analysis",
)

analysis

<xarray.Dataset> Size: 10GB
Dimensions:                   (point: 195605, time: 909)
Coordinates:
  * point                     (point) int64 2MB 0 1 2 3 ... 195602 195603 195604
  * time                      (time) datetime64[ns] 7kB 2021-04-18T14:45:00 ....
Data variables:
    x                         (point) float64 2MB ...
    y                         (point) float64 2MB ...
    z                         (point) float64 2MB ...
    m3c2_distance             (point, time) float64 1GB ...
    smoothed_distance         (point, time) float64 1GB ...
    uncertainty_lodetection   (point, time) float64 1GB ...
    uncertainty_spread1       (point, time) float64 1GB ...
    uncertainty_num_samples1  (point, time) int64 1GB ...
    uncertainty_spread2       (point, time) float64 1GB ...
    uncertainty_num_samples2  (point, time) int64 1GB ...
Attributes:
    coordinates_complete:           true
    schema_name:                    4D-OBC HDF5 analysis
    schema_version:                 0.1.0
    dataset_id:                     FinRiverbank
    point_index_semantics:          Original py4dgeo core-point index
    time_index_semantics:           Original py4dgeo time index
    obc_end_epoch_semantics:        Inclusive
    crs:                            EPSG:25832
    smoothed_distance_available:    true
    distance_for_compute_variable:  smoothed_distance
    uncertainties_complete:         true
    conversion_complete:            true

In [6]:
obcs = xr.open_dataset(
    output_h5,
    engine="h5netcdf",
    group="obcs",
)

obcs

<xarray.Dataset> Size: 35MB
Dimensions:               (object: 1337, object_offset: 1338,
                           membership: 2033996, point_offset: 195606,
                           point_membership: 2033996)
Dimensions without coordinates: object, object_offset, membership,
                                point_offset, point_membership
Data variables: (12/23)
    object_id             (object) <U26 139kB ...
    display_id            (object) <U23 123kB ...
    n_points              (object) int32 5kB ...
    object_point_offsets  (object_offset) int64 11kB ...
    start_epoch           (object) int32 5kB ...
    end_epoch             (object) int32 5kB ...
    ...                    ...
    ymax                  (object) float64 11kB ...
    zmax                  (object) float64 11kB ...
    object_point_indices  (membership) int32 8MB ...
    object_distance       (membership) float64 16MB ...
    point_object_offsets  (point_offset) int64 2MB ...
    point_object_indices  (point_membership) int32 8MB ...
Attributes:
    schema_name:                4D-OBC HDF5 OBC table
    schema_version:             0.1.0
    dataset_id:                 FinRiverbank
    membership_representation:  CSR-like object-to-point index
    obc_end_epoch_semantics:    Inclusive
    object_id_scheme:           ULID
    display_id_semantics:       dataset-local, not a global identifier
    object_distance_stored:     true
    obcs_written:               1337
    conversion_complete:        true
    reverse_index_stored:       true

In [ ]:
print("Points:", analysis.sizes["point"])
print("Times:", analysis.sizes["time"])
print("OBCs:", obcs.sizes["object"])
print("Memberships:", obcs.sizes["membership"])

### Detail explanation of each variable:

| Group | Dataset | Dimensions | Default? | Definition |
|---|---|---|---|---|
| `/analysis` | `point` | `(point,)` | Yes | Original py4dgeo core-point index. This is the stable point axis used throughout the file. |
| `/analysis` | `x` | `(point,)` | Yes | X coordinate of each core point. |
| `/analysis` | `y` | `(point,)` | Yes | Y coordinate of each core point. |
| `/analysis` | `z` | `(point,)` | Yes | Z coordinate of each core point. |
| `/analysis` | `time` | `(time,)` | Yes | Observation timestamps stored as integer nanoseconds since `1970-01-01 00:00:00`. Xarray reconstructs them as datetime coordinates. |
| `/analysis` | `m3c2_distance` | `(point, time)` | Yes | Raw M3C2 surface-change distance for every core point and time step. This is the main scientific change array. |
| `/analysis` | `smoothed_distance` | `(point, time)` | If available/configured | Temporally smoothed M3C2 distance from py4dgeo. Stored when `include_smoothed_distances=True` and the source analysis contains it. |
| `/analysis` | `uncertainty_lodetection` | `(point, time)` | Yes, by default | M3C2 level-of-detection threshold. Useful for determining whether the magnitude of an observed change exceeds estimated uncertainty. |
| `/analysis` | `uncertainty_spread1` | `(point, time)` | Optional | Local spread of the first point cloud contributing to the M3C2 uncertainty estimate. Stored only when selected in `uncertainty_fields` or when all uncertainty fields are requested. |
| `/analysis` | `uncertainty_num_samples1` | `(point, time)` | Optional | Number of samples from the first point cloud contributing to the local M3C2 estimate. |
| `/analysis` | `uncertainty_spread2` | `(point, time)` | Optional | Local spread of the second point cloud contributing to the M3C2 uncertainty estimate. |
| `/analysis` | `uncertainty_num_samples2` | `(point, time)` | Optional | Number of samples from the second point cloud contributing to the local M3C2 estimate. |
| `/obcs` | `object_id` | `(object,)` | Yes | Globally unique ULID assigned to each 4D-OBC. This should be the persistent identifier used by annotations, ML splits, databases, and releases. |
| `/obcs` | `display_id` | `(object,)` | Yes | Human-readable dataset-local identifier, e.g. `FinRiverbank-obc-000417`. Intended for notebooks, UI, figures, and presentation. |
| `/obcs` | `source_object_index` | `(object,)` | Yes | Original position of the OBC in `analysis.objects` in the source py4dgeo archive. Provides provenance back to the source ZIP. |
| `/obcs` | `n_points` | `(object,)` | Yes | Number of core points belonging to each OBC. Equivalent to the difference between two consecutive `object_point_offsets`. Stored for convenience. |
| `/obcs` | `start_epoch` | `(object,)` | Yes | First time index at which the OBC is active. |
| `/obcs` | `end_epoch` | `(object,)` | Yes | Last time index at which the OBC is active. The end epoch is inclusive. |
| `/obcs` | `seed_point_index` | `(object,)` | Yes | Core-point index of the seed used to initiate the py4dgeo OBC region-growing process. |
| `/obcs` | `seed_start_epoch` | `(object,)` | Yes | Start epoch associated with the OBC seed. |
| `/obcs` | `seed_end_epoch` | `(object,)` | Yes | End epoch associated with the OBC seed. |
| `/obcs` | `threshold` | `(object,)` | Yes | Threshold used by py4dgeo for OBC region growing / time-series similarity. |
| `/obcs` | `centroid_x` | `(object,)` | Yes | Mean X coordinate of all core points belonging to the OBC. |
| `/obcs` | `centroid_y` | `(object,)` | Yes | Mean Y coordinate of all core points belonging to the OBC. |
| `/obcs` | `centroid_z` | `(object,)` | Yes | Mean Z coordinate of all core points belonging to the OBC. |
| `/obcs` | `xmin` | `(object,)` | Yes | Minimum X coordinate of the OBC's member points. |
| `/obcs` | `ymin` | `(object,)` | Yes | Minimum Y coordinate of the OBC's member points. |
| `/obcs` | `zmin` | `(object,)` | Yes | Minimum Z coordinate of the OBC's member points. |
| `/obcs` | `xmax` | `(object,)` | Yes | Maximum X coordinate of the OBC's member points. |
| `/obcs` | `ymax` | `(object,)` | Yes | Maximum Y coordinate of the OBC's member points. |
| `/obcs` | `zmax` | `(object,)` | Yes | Maximum Z coordinate of the OBC's member points. |
| `/obcs` | `object_point_offsets` | `(object_offset,)` | Yes | CSR-style offsets defining where each object's membership begins and ends inside `object_point_indices`. Length is `number_of_objects + 1`. |
| `/obcs` | `object_point_indices` | `(membership,)` | Yes | Flattened list of all core-point memberships of all OBCs. Together with `object_point_offsets`, this is the canonical OBC → point membership representation. |
| `/obcs` | `object_distance` | `(membership,)` | Optional, off by default | Per-member py4dgeo OBC region-growing/time-series distance. Corresponds element-for-element with `object_point_indices`. This is **not** the M3C2 distance. |
| `/obcs` | `point_object_offsets` | `(point_offset,)` | Optional, off by default | Reverse-index offsets for querying core point → OBC memberships. Length is `number_of_points + 1`. |
| `/obcs` | `point_object_indices` | `(point_membership,)` | Optional, off by default | Flattened OBC indices used by the optional reverse point → OBC index. This duplicates information already contained in the canonical object → point membership representation. |

### Using a high-level API for OBCH5 data.

In [ ]:
data = OBCH5(output_h5)
data

In [ ]:
objects = data.object_table()
objects.head()

## 6. Query examples

### Query a few point time series

In [ ]:
points = data.query_points(
    [100, 200, 300],
    variables=[
        "m3c2_distance",
        "smoothed_distance",
        "uncertainty_lodetection",
    ],
)

points[["m3c2_distance","smoothed_distance"]]

In [ ]:
arr = points[
    [
        "m3c2_distance",
        "smoothed_distance",
        "uncertainty_lodetection",
    ]
].to_array(dim="variable")

arr.plot.line(
    x="time",
    hue="variable",
    col="point",
    col_wrap=1,
)

### Spatial bbox query

In [ ]:
subset = data.query_bbox(
    xmin=analysis.x.min().item(),
    xmax=analysis.x.min().item() + 20,
    ymin=analysis.y.min().item(),
    ymax=analysis.y.min().item() + 20,
)

subset

In [ ]:
print("Selected points:", subset.sizes["point"])
subset["smoothed_distance"].mean("point").plot()

### Polygon query

In [ ]:
analysis.y.min(), analysis.x.min(), analysis.y.max(), analysis.x.max()

In [ ]:
xmin = 539600.857
xmax = xmin + 10
ymin = 7757300.335
ymax = ymin + 10

polygon = box(
    xmin,
    ymin,
    xmax,
    ymax,
)

subset = data.query_polygon(polygon)

subset

### Query one 4D-OBC

In [ ]:
obc_id = objects.iloc[8]["object_id"]
obc_id

In [ ]:
obc = data.query_obc(
    obc_id,
    variables=[
        "m3c2_distance",
        "smoothed_distance",
        "uncertainty_lodetection",
        "x",
        "y",
        "z"
    ],
    )

obc

In [ ]:
obc["smoothed_distance"].mean("point").plot()

In [ ]:
# plot 
import matplotlib.pyplot as plt

fig, (ax_space, ax_time) = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

# Spatial distribution
sc = ax_space.scatter(
    obc["x"],
    obc["y"],
    c=obc["z"],
    s=8,
)

ax_space.set_xlabel("X")
ax_space.set_ylabel("Y")
ax_space.set_aspect("equal")
ax_space.set_title("OBC core points")

fig.colorbar(
    sc,
    ax=ax_space,
    label="Z",
)


# Temporal behaviour
ts = obc["smoothed_distance"]

for p_ts in ts:
    ax_time.plot(
        obc["time"],
        p_ts,
        alpha=0.3,
        color="C0",
    )

ax_time.axhline(0, linewidth=0.8)
ax_time.set_xlabel("Time")
ax_time.tick_params(axis="x", labelrotation=30)
ax_time.set_ylabel("Smoothed M3C2 distance")
ax_time.set_title("OBC temporal behaviour")
ax_time.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_space, ax_time) = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

# Spatial distribution
sc = ax_space.scatter(
    obc["x"],
    obc["y"],
    c=obc["z"],
    s=8,
)

ax_space.set_xlabel("X")
ax_space.set_ylabel("Y")
ax_space.set_aspect("equal")
ax_space.set_title("OBC core points")

fig.colorbar(
    sc,
    ax=ax_space,
    label="Z",
)


# Temporal behaviour
ts = obc["smoothed_distance"]

mean = ts.mean("point")
q25 = ts.quantile(0.05, dim="point")
q75 = ts.quantile(0.95, dim="point")

ax_time.plot(
    obc["time"],
    mean,
    label="Mean",
)

ax_time.fill_between(
    obc["time"],
    q25,
    q75,
    alpha=0.25,
    label="5–95%",
)

ax_time.axhline(0, linewidth=0.8)
ax_time.set_xlabel("Time")
ax_time.tick_params(axis="x", labelrotation=30)
ax_time.set_ylabel("Smoothed M3C2 distance")
ax_time.set_title("OBC temporal behaviour")
ax_time.legend()

plt.tight_layout()
plt.show()

### OBC membership and corepoints

object_point_offsets
    tells us where an object's point list begins and ends.

object_point_indices
    contains the flattened core-point memberships.

In [ ]:
i = 0

offsets = obcs["object_point_offsets"].values
indices = obcs["object_point_indices"].values

start = offsets[i]
end = offsets[i + 1]

point_indices = indices[start:end]

print("Number of points:", len(point_indices))
print(point_indices[:10])

### Number of active OBCs at a given time

In [ ]:
active = data.obcs_at_time(100)

print(f"{len(active)} OBCs are active at time index 100")

active.head()


### Analysis-ready data samples

In [ ]:
sample = data.ml_sample_for_obc(obc_id)

sample.keys()

x:
    M3C2 time-series tensor, typically [n_points, n_times].

xyz:
    Core-point coordinates [n_points, 3].

object_id:
    Stable identifier used to connect predictions back to labels/metadata.

In [ ]:
print(sample["x"].shape)
print(sample["xyz"].shape)

### Direct access using xarray

In [ ]:
analysis["m3c2_distance"].isel(
    point=slice(200, 300)
).mean("point").plot()